In [52]:
#Building an rnn from scratch
import torch
import torch.nn as nn

class RNN(nn.Module):  # forula : h_t = tanh( W_x · x_t  +  W_h · h_{t-1}  +  b )          x_t(input_size,)
    def __init__(self,input_size=4,hidden_size=3):                                         #h_t(hidden_size,)
        super().__init__()
        self.hidden_size = hidden_size
        
        self.W_x = nn.Parameter(torch.randn((hidden_size,input_size)))
        self.W_h =nn.Parameter(torch.randn((hidden_size,hidden_size)))
        self.b = nn.Parameter(torch.randn((hidden_size,)))
        
    
    
    def forward(self,x):                                                                   #x(B,T,input_size)
        B = x.shape[0]
        T = x.shape[1]
        h_t = torch.zeros((B,self.hidden_size))

        outputs_list = []
        for t in range(T):
            
            h_t = torch.tanh( x[:,t,:]  @ self.W_x.T +  h_t @self.W_h.T +self.b)
            outputs_list.append(h_t)

        outputs = torch.stack(outputs_list,dim=1)
        h_final = h_t

        return outputs,h_final



In [53]:
#Building an LSTM from scratch 
#formula:
#c_t = f_t * c_{t-1}  +  i_t * g_t  
#h_t = o_t * tanh(c_t) 

#f_t = σ(x_t W_xf + h_{t-1} W_hf + b_f)
#i_t = σ(x_t W_xi + h_{t-1} W_hi + b_i)
#g_t = tanh(x_t W_xg + h_{t-1} W_hg + b_g)
#o_t = σ(x_t W_xo + h_{t-1} W_ho + b_o)

  
class LSTM(nn.Module):
    def __init__(self,input_size,hidden_size):
        super().__init__()

        self.H = hidden_size

        #f_t parameters
        self.Wf_x = nn.Parameter(torch.randn((hidden_size,input_size)) *0.1)
        self.Wf_h =nn.Parameter(torch.randn((hidden_size,hidden_size))*0.1)
        self.bf = nn.Parameter(torch.randn((hidden_size,))*0.1)

        #i_t parameters
        self.Wi_x = nn.Parameter(torch.randn((hidden_size,input_size))*0.1)
        self.Wi_h =nn.Parameter(torch.randn((hidden_size,hidden_size))*0.1)
        self.bi = nn.Parameter(torch.randn((hidden_size,))*0.1)

        #g_t parameters
        self.Wg_x = nn.Parameter(torch.randn((hidden_size,input_size))*0.1)
        self.Wg_h =nn.Parameter(torch.randn((hidden_size,hidden_size))*0.1)
        self.bg = nn.Parameter(torch.randn((hidden_size,))*0.1)

        #o_t parameters
        self.Wo_x = nn.Parameter(torch.randn((hidden_size,input_size))*0.1)
        self.Wo_h =nn.Parameter(torch.randn((hidden_size,hidden_size))*0.1)
        self.bo = nn.Parameter(torch.randn((hidden_size,))*0.1)

    def forward(self,x):
        B = x.shape[0]
        T = x.shape[1]
        outputs_list = []
        h_t , c_t = torch.zeros((B,self.H)) , torch.zeros((B,self.H)) #at t = 0
        for t in range(T):
            x_t = x[:,t,:]
            f_t = torch.sigmoid(x_t @ self.Wf_x.T + h_t @ self.Wf_h.T + self.bf)
            i_t = torch.sigmoid(x_t @ self.Wi_x.T + h_t @ self.Wi_h.T + self.bi)
            g_t = torch.tanh(x_t @ self.Wg_x.T + h_t @ self.Wg_h.T + self.bg)
            c_t = f_t * c_t + i_t * g_t
            o_t = torch.sigmoid(x_t @ self.Wo_x.T + h_t @ self.Wo_h.T + self.bo)
            h_t = o_t * torch.tanh(c_t)
            outputs_list.append(h_t)
        outputs = torch.stack(outputs_list,dim=1)
        return outputs, (h_t, c_t)

In [54]:
#Practiciing with the reversal problem  ("abc"    →  "cba")

CHARS = list("abcdefghijklmnopqrstuvwxyz")
itos = ["<pad>", "<sos>","<eos>"] + CHARS
stoi = {s:i for i,s in enumerate(itos)}

In [55]:
s = 'hi'
s[::-1]

'ih'

In [56]:
#create the dataset
import random
def make_pair(min_len = 2 , max_len = 18 ):
    s = "".join(random.choices(CHARS,k=random.randint(min_len,max_len)))
    return s,s[::-1]

In [57]:
print(make_pair())

('fgpxumrhjx', 'xjhrmuxpgf')


In [58]:
def encode(s,add_sos_eos = False):
    ids = [stoi[c] for c in s]
    return [stoi["<sos>"]] + ids + [stoi["<eos>"]] if add_sos_eos else ids

In [59]:
#make batch data
import torch
from torch.nn.utils.rnn import pad_sequence


def make_batch(B):
    src_list = []
    tgt_list=[]

    for _ in range(B):
        src,tgt = make_pair()
        src = encode(src,add_sos_eos=False)
        tgt = encode(tgt , True)
        src_list.append(torch.tensor(src))
        tgt_list.append(torch.tensor(tgt))

    srcs = pad_sequence(src_list,batch_first=True,padding_value=stoi['<pad>'])
    tgts = pad_sequence(tgt_list,batch_first=True,padding_value=stoi['<pad>'])
    
    return srcs,tgts

In [60]:
s,t = make_batch(4)

In [61]:
s,t

(tensor([[ 4, 10, 23, 26, 20,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [13,  3,  5, 22, 20, 27,  6,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 9, 27, 17, 27, 14, 15, 17, 23, 13,  4, 20, 27, 21,  5, 16, 21],
         [13, 18, 25, 25,  8, 21, 24, 23, 25, 22,  6,  0,  0,  0,  0,  0]]),
 tensor([[ 1, 20, 26, 23, 10,  4,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 1,  6, 27, 20, 22,  5,  3, 13,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 1, 21, 16,  5, 21, 27, 20,  4, 13, 23, 17, 15, 14, 27, 17, 27,  9,  2],
         [ 1,  6, 22, 25, 23, 24, 21,  8, 25, 25, 18, 13,  2,  0,  0,  0,  0,  0]]))

In [62]:
#builing the encoder 

class Encoder(nn.Module):
    def __init__(self,vocab_size,emb_size,hidden_size,num_layers,dropout):
        super().__init__()

        self.embeddings = nn.Embedding(vocab_size,emb_size,padding_idx=stoi['<pad>'])
        self.dropout = nn.Dropout(p=dropout)

        self.lstm = nn.LSTM(input_size=emb_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers>1 else 0)               #(B, T, emb_dim) → outputs (B, T, H) + (h_n, c_n) each (L, B, H)



    def forward(self,src) : 
        emb = self.embeddings(src)
        embedded = self.dropout(emb)
        outputs , (h_n , c_n) = self.lstm(embedded)

        return outputs , (h_n , c_n)

In [63]:
#NOTE : 
#for 2 layers :  outputs[:,-1]  has h_n of the second layer 
#                h[-1] has h_n of the first layer  (not in outputs at all )


In [64]:
#building the decoder 
class Decoder(nn.Module):
    def __init__(self,vocab_size,emb_size,hidden_size,num_layers,dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size,emb_size,padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(input_size=emb_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers>1 else 0) 

        self.linear = nn.Linear(hidden_size,vocab_size)

    def forward(self,input,hidden,cell):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        output, (h,c) = self.lstm(embedded,(hidden,cell))
        logits = self.linear(output.squeeze(1))
        return logits , (h,c)


In [65]:

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.enc = encoder
        self.dec = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):         
        B, T = tgt.shape
        V = self.dec.linear.out_features                        

        outputs = torch.zeros(B, T, V, device=src.device)          

        _, (h, c) = self.enc(src)                                   
        input = tgt[:, 0]                                   

        for t in range(1, T):                                        # FIX 3: start at 1
            logits, (h, c) = self.dec(input, h, c)                   # FIX 2: self.dec, not dec
            outputs[:, t] = logits                                   # prediction that should equal tgt[:, t]

            use_teacher = random.random() < teacher_forcing_ratio
            input = tgt[:, t] if use_teacher else logits.argmax(1)   # FIX 3: tgt[:, t], not t+1

        return outputs

In [66]:
VOCAB_SIZE = len(itos)
model = Seq2Seq(Encoder(VOCAB_SIZE, 32, 128, 1, 0.1),
                Decoder(VOCAB_SIZE, 32, 128, 1, 0.1))

optimizer = torch.optim.Adam(model.parameters() , lr =18e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)         #CrossEntropyLoss that ignores padding (ID 0)
N_STEPS = 30000

# ---- training loop ----
for step in range(1, N_STEPS + 1):
    model.train()                                   # dropout ON

    src, tgt = make_batch(64)

    optimizer.zero_grad()                           # 1. clear old gradients
    outputs = model(src,tgt,0.5)

    V = outputs.shape[-1]
    pred    = outputs[:,1:].reshape(-1,V)
    answers = tgt[:,1:].reshape(-1)
    loss = criterion(pred, answers)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   #clip gradients to max norm 1.0
    optimizer.step()

    if step % 200 == 0:
        print(f"step {step:4d} | loss {loss.item():.3f}")

step  200 | loss 1.565
step  400 | loss 1.143
step  600 | loss 0.916
step  800 | loss 0.881
step 1000 | loss 0.680
step 1200 | loss 0.664
step 1400 | loss 0.651
step 1600 | loss 0.629
step 1800 | loss 0.559
step 2000 | loss 0.634
step 2200 | loss 0.612
step 2400 | loss 0.641
step 2600 | loss 0.434
step 2800 | loss 0.600
step 3000 | loss 0.463
step 3200 | loss 0.511
step 3400 | loss 0.578
step 3600 | loss 0.550
step 3800 | loss 0.705
step 4000 | loss 0.430
step 4200 | loss 0.429
step 4400 | loss 0.410
step 4600 | loss 0.516
step 4800 | loss 0.472
step 5000 | loss 0.595
step 5200 | loss 0.412
step 5400 | loss 0.591
step 5600 | loss 0.416
step 5800 | loss 0.470
step 6000 | loss 0.542
step 6200 | loss 0.525
step 6400 | loss 0.465
step 6600 | loss 0.590
step 6800 | loss 0.495
step 7000 | loss 0.425
step 7200 | loss 0.502
step 7400 | loss 0.456
step 7600 | loss 0.657
step 7800 | loss 0.669
step 8000 | loss 0.398


KeyboardInterrupt: 

In [67]:
@torch.no_grad()
def translate(model, s, max_len=50):
    model.eval()

    src = torch.tensor([encode(s)])     #encode s (no <sos>/<eos>), make it a tensor of shape (1, len(s))
    _, (h, c) =model.enc(src)       #run the encoder

    input = torch.tensor([stoi['<sos>']])   # tensor of shape (1,) containing the <sos> ID
    result = []
    for _ in range(max_len):
        logits, (h, c) = model.dec(input,h,c) 
        next_id = logits.argmax(1).item()
        if next_id == stoi['<eos>']:
            break
        result.append(itos[next_id])
        input = torch.tensor([next_id]) # next_id as a tensor of shape (1,)

    return "".join(result)

In [71]:
import random

def make_test_set(n, min_len, max_len, seed):
    rng = random.Random(seed)                      # own random generator → same strings every run
    pairs = []
    for _ in range(n):
        k = rng.randint(min_len, max_len)
        s = "".join(rng.choices(CHARS, k=k))
        pairs.append((s, s[::-1]))
    return pairs

# same lengths as training (3–8), never seen before
test_normal = make_test_set(500, 3, 8,  seed=123)

# LONGER than anything seen in training (10–12)
test_long   = make_test_set(200, 10, 12, seed=456)

print(test_normal[:3])
print(test_long[:3])

[('gtg', 'gtg'), ('joi', 'ioj'), ('eii', 'iie')]
[('lmkxbtxndmtc', 'ctmdnxtbxkml'), ('mgdeknusypsy', 'yspysunkedgm'), ('bnjzsqwfudl', 'ldufwqszjnb')]


In [72]:
def evaluate(model, pairs, show_errors=5):
    correct = 0
    errors = []
    for src, expected in pairs:
        pred = translate(model, src)
        if pred == expected:
            correct += 1
        elif len(errors) < show_errors:
            errors.append((src, expected, pred))

    acc = correct / len(pairs)
    print(f"exact-match accuracy: {acc:.1%}  ({correct}/{len(pairs)})")
    for src, expected, pred in errors:
        print(f"  {src:14s} expected {expected:14s} got {pred}")
    return acc

print("=== lengths 3–8 (same as training) ===")
evaluate(model, test_normal)

print("\n=== lengths 10–12 (longer than training) ===")
evaluate(model, test_long)

=== lengths 3–8 (same as training) ===


exact-match accuracy: 85.0%  (425/500)
  joi            expected ioj            got oij
  oyaxqlo        expected olqxayo        got olqoayo
  xre            expected erx            got esx
  xzk            expected kzx            got kkx
  sxqzzqg        expected gqzzqxs        got gqzqzxs

=== lengths 10–12 (longer than training) ===
exact-match accuracy: 32.5%  (65/200)
  lmkxbtxndmtc   expected ctmdnxtbxkml   got ctmdnxtbmmml
  mgdeknusypsy   expected yspysunkedgm   got ysppyudekygm
  qiauhepkcmot   expected tomckpehuaiq   got tomckpekuaiq
  ehmmgdwjcm     expected mcjwdgmmhe     got mjcwdgmmhe
  dochzidberz    expected zrebdizhcod    got zrebdihcood


0.325